In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import lxml
from selenium import webdriver
import time


In [33]:
# %pip install lxml
# %pip install selenium

In [2]:
urls = {
    "US": "https://www.ysl.com/en-us/ca/shop-women/handbags",
    "FR": "https://www.ysl.com/en-fr/shop-women/handbags",
    "IT": "https://www.ysl.com/en-it/shop-women/handbags",
    "CN": "https://www.ysl.cn/categories/shop-women/handbags/view-all.html"
}

driver = webdriver.Chrome() # use selenium for javascript -- prices won't load with just soup

In [19]:
def scroll_to_bottom(driver):

    last_height = driver.execute_script("return document.body.scrollHeight")

    while True:

        # scroll in small steps
        for i in range(10):
            driver.execute_script("window.scrollBy(0, 600)")
            time.sleep(0.5)

        new_height = driver.execute_script("return document.body.scrollHeight")

        if new_height == last_height:
            break

        last_height = new_height

In [20]:
data = []

for country, url in urls.items():
    driver.get(url)
    time.sleep(5)

    scroll_to_bottom(driver) # to get all the bags

    html = driver.page_source
    soup = BeautifulSoup(html, "html.parser")

    # china html is different
    if country == "CN":
        titles = soup.select(".productName")
        prices = soup.select(".productPrice")
    else:
        titles = soup.select('h2[data-qa^="plp-product-title"]')
        prices = soup.select('span[data-qa^="plp-product-price"]')

    # was for debugging
    # print("titles:", len(titles))
    # print("prices:", len(prices))

    for t,p in zip(titles, prices):
        name = t.get_text(strip=True)
        priceText = p.get_text(strip=True)
        price = float(re.sub(r"[^\d.]", "", priceText))

        if country == "CN": # china product ID is also stored differently - in the product URL only
            link = t.find_parent("a")["href"]
            sku = link.split("/")[-1].split("?")[0].replace(".html", "").upper()
        else:
            sku = t["data-qa"].replace("plp-product-title-", "")

        currency = re.findall(r"[^\d.,\s]", priceText)[0]

        data.append({
        "brand": "Saint Laurent",
        "product_name": name,
        "price": price,
        "currency": currency,
        "country": country,
        "sku": sku
})


In [21]:
df = pd.DataFrame(data)
print(df)
print(len(df))


              brand               product_name    price currency country  \
0     Saint Laurent   MOMBASA small in leather   3450.0        $      US   
1     Saint Laurent   MOMBASA small in leather   3450.0        $      US   
2     Saint Laurent   MOMBASA small in leather   3450.0        $      US   
3     Saint Laurent  MOMBASA medium in leather   4300.0        $      US   
4     Saint Laurent  MOMBASA medium in leather   4300.0        $      US   
...             ...                        ...      ...      ...     ...   
1436  Saint Laurent          KATE流苏中号鳄鱼纹压纹皮革手袋  22500.0        ¥      CN   
1437  Saint Laurent          KATE流苏中号鳄鱼纹压纹皮革手袋  22500.0        ¥      CN   
1438  Saint Laurent             KATE流苏小号粒面皮革手袋  18500.0        ¥      CN   
1439  Saint Laurent          KATE流苏小号鳄鱼纹压纹皮革手袋  19900.0        ¥      CN   
1440  Saint Laurent          KATE流苏小号鳄鱼纹压纹皮革手袋  19900.0        ¥      CN   

                  sku  
0     851432AAGWJ1000  
1     851432AAGWK2050  
2     851432AAG

In [34]:
df.to_csv("data/ysl_allbags.csv", index=False)

In [35]:
pivot_df = df.pivot_table(index="sku", columns="country", values="price", aggfunc="first")

num_all_4 = (pivot_df.notna().sum(axis=1) == 4).sum()
print(num_all_4)

171


In [36]:
all4 = pivot_df[pivot_df.notna().all(axis=1)]
all4

country,CN,FR,IT,US
sku,,,,
364021BOW0J1000,18500.0,1950.0,1950.0,2450.0
364021BOW0J9207,18500.0,1950.0,1950.0,2450.0
37829902G9W1000,28000.0,2950.0,2950.0,3600.0
378299B681N1000,26000.0,2800.0,2800.0,3400.0
39203502G9W1000,20500.0,2200.0,2200.0,2700.0
...,...,...,...,...
862712AAB326195,23000.0,2500.0,2500.0,3000.0
871329AAANG1000,21500.0,2300.0,2300.0,2900.0
871329AAANG2721,21500.0,2300.0,2300.0,2900.0


In [37]:
sample_skus = all4.sample(10, random_state=21).index
df_40 = df[df["sku"].isin(sample_skus)].copy()
df_40 = df_40.reset_index(drop=True)
df_40


,brand,product_name,price,currency,country,sku
0,Saint Laurent,MOMBASA small in leather,3450.0,$,US,851432AAGWK6195
1,Saint Laurent,LE 5 À 7 supple small IN GRAINED LEATHER,2750.0,$,US,713938AAAUQ1000
2,Saint Laurent,LE 5 À 7 supple small in suede,2900.0,$,US,8505331U80W1997
3,Saint Laurent,Y small tote in leather,3500.0,$,US,835274AAEC36359
4,Saint Laurent,JAMIE SHOPPING small in lambskin,3800.0,$,US,833948AAB329295
5,Saint Laurent,GABY vanity in lambskin,2300.0,$,US,7667311EL072536
6,Saint Laurent,PUFFER SMALL in lambskin,3300.0,$,US,5774761EL071000
7,Saint Laurent,SAC DE JOUR IN SMOOTH LEATHER - SMALL,3600.0,$,US,37829902G9W1000
8,Saint Laurent,ENVELOPE SMALL IN QUILTED GRAIN DE POUDRE EMBO...,2650.0,$,US,600195BOW911000
9,Saint Laurent,Shopping Saint Laurent in leather,1550.0,$,US,600281CSV0J2721


In [38]:
df_40.to_csv("data/ysl_10bags_4countries_cleaned.csv", index=False)

In [43]:
# TODO:

# it's repeating bags that are the same style + price, but just the color is diff - should we consolidate b4 picking 10
# convert everything to USD